In [7]:
import os 
from dotenv import load_dotenv

load_dotenv()

# langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.embeddings import JinaEmbeddings

In [25]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_community.embeddings import JinaEmbeddings

# Load env variables
load_dotenv(override=True)

# 2. LLM Definition
llm = ChatOpenAI(
    model=os.getenv("EVREN_MODEL"),          # deepseek-v4-flash
    api_key=os.getenv("EVREN_API_KEY"),     
    base_url=os.getenv("EVREN_BASE_URL"),
    temperature=0    # https://evren-llmapi.ssyz.org.tr/v1
)

# 3. Jina Embeddings Definition
embeddings = JinaEmbeddings(
    jina_api_key=os.getenv("JINA_API_KEY"),
    model_name="jina-embeddings-v3"
)

# --- TEST ---

# LLM Test
response = llm.invoke([HumanMessage(content="Hi!")])
print("LLM Response:", response.content)

# Jina Embeddings Test
query_vector = embeddings.embed_query("HR Policy Test")
print(f"Embedding Vector Size: {len(query_vector)}")


LLM Response: Hi there! How can I help you today?
Embedding Vector Size: 1024


In [9]:
DATA_FILE_PATH = os.path.join("data" , "hr_policy.txt")

In [10]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: data\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2595

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Velune - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried


In [11]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Velune - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of emer

In [12]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [13]:
print(f"Total characters in document: {len(documents[0].page_content)}")

Total characters in document: 2595


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nVelune - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to

In [15]:
len(chunks)

9

In [16]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Velune - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [17]:
print(chunks[5])

page_content='5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.' metadata={'source': 'data\\hr_policy.txt'}


In [18]:
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


## Vector DB

In [20]:
from langchain_community.vectorstores import FAISS 

vector_store = FAISS.from_documents(chunks , embeddings)

print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [21]:
test_query = "How many sick leaves employees get"

## Similarity Search 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



## Tool

In [22]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

## Retrieval

In [24]:
tr = llm.invoke("Hey what is the leave policy")
tr.content

"A leave policy is a set of rules that explains what time off employees can take from work and how it works. It usually covers:\n\n- **Paid time off (PTO)** – vacation or personal days\n- **Sick leave** – when you're ill or need medical care\n- **Public holidays** – days the company is closed\n- **Parental leave** – maternity, paternity, or adoption leave\n- **Unpaid leave** – time off without pay\n- **Emergency/compassionate leave** – for urgent personal matters\n- **How to request leave** – notice period, approval process\n- **Accrual rules** – how much leave you earn and when\n\nIf you’re asking about a specific company or country, let me know and I can give a more detailed answer!"

In [26]:
llm.model_name

'deepseek-v4-flash'

## Create a Agent

In [27]:
from langchain.agents import create_agent 

In [28]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Velune. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [29]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [31]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)

print(response)

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='cf38cb24-4ac7-4d95-94d6-5e2b9d86a24d'), AIMessage(content='I work for **Velune** as your friendly HR assistant! 😊\n\nHow can I help you today? I can look up information about leave, work-from-home policies, probation, notice periods, reimbursements, code of conduct, holidays, or the exit process. Just let me know what you need!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 145, 'prompt_tokens': 444, 'total_tokens': 589, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 80, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256, 'image_tokens': None, 'text_tokens': None, 'created_cache_tokens': 0, 'multimodal_tokens': None}, 'evren': {'credits_held_cr': '0.0000', 'cred

In [32]:
response["messages"][-2].content

'tell me which org you work for'

In [33]:
print(response["messages"][-1].content)

I work for **Velune** as your friendly HR assistant! 😊

How can I help you today? I can look up information about leave, work-from-home policies, probation, notice periods, reimbursements, code of conduct, holidays, or the exit process. Just let me know what you need!
